# Third model family: InternVL3 on FERMAT

Twelfth notebook. LLaVA-NeXT confirmed the reasoning result cross-family
but was capability-gated on perception (0.8% free-response OCR accuracy).
This notebook re-attempts both arms on **InternVL3-8B**, trained from
scratch by Shanghai AI Lab/OpenGVLab -- not a Qwen fine-tune (unlike
MiniCPM-V and olmOCR, both literally built on Qwen2-VL, which were ruled
out for this reason) -- at the identical $n=300$ balanced sample every
reference run in this project has used.

## Revision note: the first version of this notebook used the wrong checkpoint

The original build used `OpenGVLab/InternVL3-8B` with
`trust_remote_code=True` and a custom `.chat()` interface, per that
checkpoint's own model card. **It crashed on the real run** with
`AttributeError: 'InternVLChatModel' object has no attribute
'all_tied_weights_keys'` -- a real, documented incompatibility between
InternVL's custom modeling code (written for `transformers` v4) and the
`transformers` v5 that a bare `%pip install transformers` now resolves to
(the same class of break has hit several other trust_remote_code
multimodal models -- Molmo, moondream, InternVisionModel -- for the exact
same reason: v5 requires custom model classes to properly initialize
`all_tied_weights_keys`, and older custom code doesn't).

**Fixed by switching to the native `transformers` integration instead of
pinning an old version**: `transformers` added InternVL support directly
into the library (verified against the current docs, not assumed), with
its own converted checkpoint, `OpenGVLab/InternVL3-8B-hf`. This uses the
standard `AutoModelForImageTextToText` + `AutoProcessor` +
`apply_chat_template` flow -- no `trust_remote_code`, no custom `.chat()`
method, no custom image-tiling code to maintain. It also turns out to have
an interface nearly identical to LLaVA-NeXT's (same message-content-list
shape, same combined `apply_chat_template(tokenize=True, return_dict=True,
...)` call, and critically, **standard `generate()` with
`num_return_sequences` support** -- the old `.chat()`-based version had no
batched sampling at all, so this fix is also a real efficiency win, not
just a bug fix.

**Prerequisite for running:** cells 2-3 (install, auth) must run this
session.


In [ ]:
# Install cell: GPU-dependent packages only.
%pip install -q transformers accelerate datasets huggingface_hub bitsandbytes


In [ ]:
# Auth & code/results access cell. Identical to notebooks 06/09/10/11.
import json
import os
from getpass import getpass

from huggingface_hub import login

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy
import pilot.canonicalize
import pilot.plotting

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")


In [ ]:
# Model load cell. Native transformers integration -- AutoModelForImageTextToText
# + AutoProcessor, no trust_remote_code. Same OOM/4-bit fallback pattern as
# notebook 10, recorded via QUANTIZED as always.
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "OpenGVLab/InternVL3-8B-hf"
QUANTIZED = False

try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    print(f"Loaded {MODEL_ID} in bfloat16 (full precision).")
except torch.cuda.OutOfMemoryError:
    print(f"bfloat16 load of {MODEL_ID} did not fit -- falling back to 4-bit "
          "quantization. This changes what is being measured; the saved "
          "results record QUANTIZED=True so this is never silently glossed over.")
    from transformers import BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    QUANTIZED = True

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

if torch.cuda.is_available():
    vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gib:.1f} GiB), "
          f"quantized={QUANTIZED}")


In [ ]:
# Sample cell. Identical to notebook 03/10 -- same function, seed, and
# balance, so this run is directly comparable to every existing reference run.
import logging

import pilot.data

logging.basicConfig(level=logging.WARNING, force=True)

N = 300
SEED = 42
TARGET_ERROR_FRAC = 0.5

sample = pilot.data.load_fermat_balanced(
    n=N, seed=SEED, target_error_frac=TARGET_ERROR_FRAC
)
N = len(sample)
n_error = sum(bool(x) for x in sample["has_error"])
print(f"{N} items, {n_error} with a mistake, {N - n_error} clean "
      f"({n_error / N:.0%} error rate)")


In [ ]:
# InternVL3 adapter cell -- REQUIRED pre-flight check before cell 6 runs.
#
# Message shape and generation call are identical to the LLaVA-NeXT
# adapter (notebook 10): no system role (folded into one user turn -- the
# InternVL docs show no system-role example either, same reasoning as
# before: don't gamble on template support that isn't demonstrated), and
# the same combined apply_chat_template(tokenize=True, return_dict=True,
# return_tensors="pt", add_generation_prompt=True) call.
import pilot.prompts


def build_internvl_messages(system_prompt: str, user_prompt: str, image) -> list[dict]:
    combined_text = f"{system_prompt}\n\n{user_prompt}"
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": combined_text},
            ],
        },
    ]


def build_internvl_grading_messages(image):
    return build_internvl_messages(
        pilot.prompts.GRADING_SYSTEM_PROMPT, pilot.prompts.GRADING_USER_PROMPT, image
    )


def build_internvl_transcription_messages(image):
    return build_internvl_messages(
        pilot.prompts.TRANSCRIPTION_SYSTEM_PROMPT, pilot.prompts.TRANSCRIPTION_USER_PROMPT, image
    )


_test_item = sample[0]
_test_messages = build_internvl_grading_messages(_test_item["image"])
_test_inputs = processor.apply_chat_template(
    _test_messages, tokenize=True, return_dict=True,
    return_tensors="pt", add_generation_prompt=True,
).to(model.device)

with torch.no_grad():
    _test_output = model.generate(**_test_inputs, max_new_tokens=64, do_sample=False)
_test_trimmed = _test_output[:, _test_inputs["input_ids"].shape[1]:]
_test_text = processor.batch_decode(_test_trimmed, skip_special_tokens=True)[0]

print("Adapter pre-flight check OK. Sample output (greedy, 64 tokens):")
print(_test_text)
assert len(_test_text.strip()) > 0, "Model produced empty output -- adapter is broken."


In [ ]:
# Grading + transcription generation, K=5 each -- mirrors notebook 10's
# structure exactly, including the batch-backoff ladder, now that the
# native integration restores num_return_sequences support (the old
# .chat()-based version had no batching at all).
import gc
import json
import os
import time

from tqdm.auto import tqdm

K_TRANSCRIPTION = 5
K_GRADING = 5
TEMP = 0.7
_BATCH_LADDER = [5, 2, 1]
_batch_state = {"index": 0}

META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
INFRA_EXCEPTIONS = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def _generate_batch(messages, n: int, temperature: float):
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=temperature, num_return_sequences=n,
        )

    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    del output_ids, inputs
    gc.collect()
    torch.cuda.empty_cache()
    return texts


def generate_k(messages, n: int, temperature: float):
    texts = []
    last_exc = None
    while len(texts) < n:
        want = n - len(texts)
        size = min(_BATCH_LADDER[_batch_state["index"]], want)
        for attempt in range(3):
            try:
                texts += _generate_batch(messages, size, temperature)
                last_exc = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                if _batch_state["index"] + 1 < len(_BATCH_LADDER):
                    _batch_state["index"] += 1
                    print(f"  OOM at batch {size}; dropping to "
                          f"{_BATCH_LADDER[_batch_state['index']]} for the rest of the run.",
                          flush=True)
                    size = min(_BATCH_LADDER[_batch_state["index"]], n - len(texts))
                    continue
                raise
            except INFRA_EXCEPTIONS as exc:
                last_exc = exc
                gc.collect()
                torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last_exc is not None:
            raise last_exc
    return texts


CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
model_slug = MODEL_ID.split("/")[-1]
checkpoint_path = (f"{CHECKPOINT_DIR}/scaleup_{model_slug}_n{N}_seed{SEED}"
                   f"_bal{int(TARGET_ERROR_FRAC * 100)}_kt{K_TRANSCRIPTION}_kg{K_GRADING}"
                   f"{'_4bit' if QUANTIZED else ''}.jsonl")

raw_results = []
if os.path.exists(checkpoint_path):
    with open(checkpoint_path) as f:
        raw_results = [json.loads(line) for line in f if line.strip()]
    valid = []
    for idx, entry in enumerate(raw_results[:N]):
        item = sample[idx]
        if not all(entry["item"].get(k) == item[k] for k in META_FIELDS):
            print(f"Checkpoint item {idx + 1} does not match sample order; resuming there.")
            break
        if (len(entry.get("transcription_samples_raw", [])) != K_TRANSCRIPTION
                or len(entry.get("grading_samples_raw", [])) != K_GRADING):
            break
        valid.append(entry)
    if len(valid) != len(raw_results):
        with open(checkpoint_path, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
        print(f"Truncated checkpoint from {len(raw_results)} to {len(valid)} valid items.")
    raw_results = valid
    print(f"Resuming from {len(raw_results)} completed items")

if len(raw_results) >= N:
    print(f"All {N} items already done.")
else:
    print(f"Starting from item {len(raw_results) + 1}/{N} "
          f"({N - len(raw_results)} remaining)", flush=True)
    with tqdm(total=(N - len(raw_results)) * (K_TRANSCRIPTION + K_GRADING),
              desc="generating", unit="sample") as pbar:
        for item_idx, item in enumerate(sample):
            if item_idx < len(raw_results):
                continue
            _t0 = time.time()
            transcription_msgs = build_internvl_transcription_messages(item["image"])
            grading_msgs = build_internvl_grading_messages(item["image"])
            transcription_texts = generate_k(transcription_msgs, K_TRANSCRIPTION, TEMP)
            pbar.update(K_TRANSCRIPTION)
            grading_texts = generate_k(grading_msgs, K_GRADING, TEMP)
            pbar.update(K_GRADING)
            _elapsed = time.time() - _t0
            entry = {
                "item": {k: item[k] for k in META_FIELDS},
                "transcription_samples_raw": transcription_texts,
                "grading_samples_raw": grading_texts,
                "quantized": QUANTIZED,
                "elapsed_seconds": _elapsed,
            }
            raw_results.append(entry)
            with open(checkpoint_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()
            print(f"  item {item_idx + 1}/{N}: {_elapsed:.1f}s "
                  f"({len(raw_results)}/{N} done)", flush=True)

print(f"raw_results: {len(raw_results)} items")


In [ ]:
# Scoring cell. Identical to notebook 10's -- same functions, same
# canonicalization path, so InternVL3's numbers are computed exactly the
# way every other model's were.
import importlib

import pilot.canonicalize
import pilot.entropy
import pilot.parsing

for m in (pilot.parsing, pilot.canonicalize, pilot.entropy):
    importlib.reload(m)

import sympy

_LATEX_OK = pilot.canonicalize.warn_if_latex_parser_missing()
_SYMPY_VERSION = sympy.__version__
print(f"LaTeX parser available: {_LATEX_OK} (sympy {_SYMPY_VERSION})")

scored_results = []
for entry in raw_results:
    item = entry["item"]

    transcription_parsed = [
        pilot.parsing.parse_transcription(t) for t in entry["transcription_samples_raw"]
    ]
    grading_parsed = [pilot.parsing.parse_grading(t) for t in entry["grading_samples_raw"]]

    transcription_answers = [
        pilot.canonicalize.canonical_answer_label(t) for t in transcription_parsed
    ]

    perception_entropy = pilot.entropy.cluster_entropy(transcription_answers)
    reasoning_entropy = pilot.entropy.cluster_entropy(
        [None if d is None else str(d) for d in grading_parsed]
    )

    majority_transcription, _ = pilot.entropy.majority_cluster(transcription_answers)
    ground_truth_answer = pilot.canonicalize.canonical_answer_label(item["pert_a"])
    transcription_correct = majority_transcription == ground_truth_answer

    majority_grading, _ = pilot.entropy.majority_cluster(
        [None if d is None else str(d) for d in grading_parsed]
    )
    grading_correct = majority_grading in {"0", "1"} and int(majority_grading) == int(
        item["has_error"]
    )

    scored_results.append({
        "orig_q": item["orig_q"],
        "pert_a": item["pert_a"],
        "has_error": item["has_error"],
        "handwriting_style": item["handwriting_style"],
        "image_quality": item["image_quality"],
        "perception_entropy": perception_entropy,
        "reasoning_entropy": reasoning_entropy,
        "transcription_correct": transcription_correct,
        "grading_correct": grading_correct,
        "n_transcription_parse_failures": sum(1 for t in transcription_parsed if t is None),
        "n_grading_parse_failures": sum(1 for d in grading_parsed if d is None),
        "all_transcription_samples_raw": entry["transcription_samples_raw"],
        "all_grading_samples_raw": entry["grading_samples_raw"],
        "model_id": MODEL_ID,
        "quantized": entry["quantized"],
        "n_items": N,
        "k_transcription": K_TRANSCRIPTION,
        "k_grading": K_GRADING,
        "target_error_frac": TARGET_ERROR_FRAC,
        "latex_parser_available": _LATEX_OK,
        "sympy_version": _SYMPY_VERSION,
    })

print(f"Scored {len(scored_results)} items.")

import pandas as pd

_df = pd.DataFrame(scored_results)
print(f"Transcription accuracy: {_df['transcription_correct'].mean():.1%}")
print(f"Grading accuracy: {_df['grading_correct'].mean():.1%} (baseline 50%)")

perception_r = pilot.plotting.bootstrap_auroc_ci(
    _df, "perception_entropy", "transcription_correct", n_boot=10000, seed=0
)
print(f"Perception AUROC: {perception_r['auroc']:.3f} "
      f"[{perception_r['ci_low']:.3f}, {perception_r['ci_high']:.3f}]")

strat = pilot.plotting.stratified_auroc(
    _df, "reasoning_entropy", "grading_correct", "has_error", n_boot=10000, seed=0
)
for level, s in strat["strata"].items():
    minority = min(s["n_error"], s["n_correct"])
    powered = minority >= pilot.plotting.SCALEUP_PREREGISTRATION["min_minority_class"]
    print(f"  has_error={level}  n={s['n_items']:3d}  n_wrong={s['n_error']:3d}  "
          f"AUROC {s['auroc']:.3f} [{s['ci_low']:.3f}, {s['ci_high']:.3f}]  "
          f"minority={minority}  {'POWERED' if powered else 'still underpowered'}")
print(f"  sign_reversal      : {strat['sign_reversal']}")
print(f"  pooled_understates : {strat['pooled_understates']}")
print()
print("Reference points: Qwen-7B perception 0.835 [0.787, 0.879];")
print("  LLaVA-NeXT perception -- gated, not reportable (0.8% free-response accuracy);")
print("  has_error=1 stratum confirmed at 0.775-0.854 across Qwen-3B/7B and LLaVA-NeXT.")


In [ ]:
# Save cell: CSV to Drive first, then repo + push. Distinct filename --
# never overwrites any prior model's results.
import subprocess
from datetime import datetime, timezone
from getpass import getpass

import pandas as pd

df = pd.DataFrame(scored_results)

model_slug_lower = MODEL_ID.split("/")[-1].lower().replace(".", "")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = (f"scaleup_n{N}_bal{int(TARGET_ERROR_FRAC * 100)}_"
            f"{'4bit_' if QUANTIZED else ''}{model_slug_lower}_{timestamp}.csv")

drive_results = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(drive_results, exist_ok=True)
df.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(df)} rows)")

_REDACT = []


def git(*args):
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add InternVL3 n=300 scale-up results: {csv_name}")
if commit.returncode != 0:
    print("git commit failed (see above) -- CSV is safe on Drive.")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is saved on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results to the repo.")
    else:
        print("Push failed (see above). The CSV is safe on Drive and in "
              "repo/results/ -- retry the push without re-running the model.")
